In [2]:
import pandas as pd
import min_features, daily_return
import importlib
importlib.reload(min_features)
importlib.reload(daily_return)

perf_df = pd.read_csv("newest_training.csv")
perf_df['Date'] = perf_df['test_start']
returns = [1, 2, 3, 5, 10]
df_daily = daily_return.pull_daily('QQQ', returns) 
return_cols = df_daily.columns[df_daily.columns.str.contains("Return_")].to_list()
df_returns = df_daily[['Date'] + return_cols][(df_daily['Date'] < '2025-12-20') & (df_daily['Date'] > '2025-01-01')].copy()

In [6]:
r = 2
df_returns = df_daily[['Date', 'Close'] + return_cols][(df_daily['Date'] < '2025-12-20') & (df_daily['Date'] > '2025-01-01')].copy()
df_returns_r = df_returns[['Date', 'Close', f'Return_{r}']].sort_values(by='Date').copy()
# df has columns: ["Date", "Return_1"] where Return_1 is 0/1 (or False/True)

s = df_returns_r[f"Return_{r}"].astype(int)

# identify streak groups (new group whenever value changes)
grp = s.ne(s.shift()).cumsum()

# streak length within each group: 1,2,3,...
streak_len = s.groupby(grp).cumcount() + 1

# positive streaks for 1s, negative for 0s
df_returns_r["streak"] = streak_len.where(s.eq(1), -streak_len)
df_returns_r["streak_lag1"] = df_returns_r["streak"].shift(1).fillna(0).astype("Int64")

#df_returns_1.sort_values(by='Date', ascending=False)

perf_cols = ['model', 'acc', 'Date', 'train_years', 'feature_set']
performance_2 = pd.merge(df_returns_r, perf_df[(perf_df['horizon'] == r) & (perf_df['test_days'] == 1)], on='Date', how='inner')

df = performance_2.copy()
# 1) Accuracy by (model, train_years, streak)
acc_piv = df.pivot_table(
    index=["model", "train_years", 'feature_set'],
    columns="streak_lag1",
    values="acc",
    aggfunc="mean"
)

# 2) Count by (model, train_years, streak)
cnt_piv = df.pivot_table(
    index=["model", "train_years", 'feature_set'],
    columns="streak_lag1",
    values="acc",          # any column works since we're counting rows
    aggfunc="size"
)

# 3) Combine into one wide table with clear column labels
out = pd.concat({"acc": acc_piv, "count": cnt_piv}, axis=1)

# optional: sort columns so streaks go -N ... -1, 1 ... N
out = out.reindex(sorted(out.columns, key=lambda x: (x[1] >= 0, x[1])), axis=1)

gcols = ["model", "train_years", "feature_set"]

df2 = df.copy()
df2["side"] = df2["streak"].gt(0).map({True: "pos", False: "neg"})  # 0 shouldn't exist with your streak logic

side_perf = (
    df2.groupby(gcols + ["side"])
       .agg(n=("acc", "size"), acc=("acc", "mean"))
       .reset_index()
)

# wide format (pos/neg columns)
side_wide = side_perf.pivot(index=gcols, columns="side", values=["acc", "n"])
side_wide

acc               n       
side                                         neg       pos   neg    pos
model         train_years feature_set                                  
random_forest 4           d+m_w/_lag    0.347368  0.759398  95.0  133.0
                          daily         0.557895  0.684211  95.0  133.0
                          daily+minute  0.284211  0.781955  95.0  133.0
                          min_w/_lag    0.189474  0.736842  95.0  133.0
                          minute        0.168421  0.774436  95.0  133.0
              6           d+m_w/_lag    0.315789  0.812030  95.0  133.0
                          daily         0.526316  0.744361  95.0  133.0
                          daily+minute  0.242105  0.827068  95.0  133.0
                          min_w/_lag    0.126316  0.834586  95.0  133.0
                          minute        0.147368  0.819549  95.0  133.0
xgboost       4           d+m_w/_lag    0.452632  0.691729  95.0  133.0
                          daily         0.526316  0.639098  95.0  133.0
                          daily+minute  0.368421  0.669173  95.0  133.0
                          min_w/_lag    0.315789  0.699248  95.0  133.0
                          minute        0.231579  0.601504  95.0  133.0
              6           d+m_w/_lag    0.421053  0.631579  95.0  133.0
                          daily         0.515789  0.714286  95.0  133.0
                          daily+minute  0.421053  0.654135  95.0  133.0
                          min_w/_lag    0.336842  0.609023  95.0  133.0
                          minute        0.315789  0.669173  95.0  133.0

In [13]:
K = 3
gcols = ["model", "train_years", "feature_set"]
streak_col = 'streak_lag1'

d = df.copy()

# keep exact -3..+3; collapse only beyond into +/-4 (representing 3+)
d["streak_bucket"] = d[streak_col].clip(lower=-K, upper=K)
d.loc[d[streak_col] < -K, "streak_bucket"] = -(K + 1)   # strictly less than -3
d.loc[d[streak_col] >  K, "streak_bucket"] =  (K + 1)   # strictly greater than +3

flip_perf = (
    d.groupby(gcols + ["streak_bucket"])
     .agg(n=("acc", "size"), acc=("acc", "mean"))
)

flip_wide = pd.concat(
    {"acc": flip_perf["acc"].unstack("streak_bucket"),
     "n":   flip_perf["n"].unstack("streak_bucket")},
    axis=1
)

# order: +1 acc, -1 acc, +1 n, -1 n, ... +3 acc, -3 acc, +3 n, -3 n, 3+ acc, -3+ acc, 3+ n, -3+ n
ordered_cols = []
for k in [1, 2, 3, "3+"]:
    pb = (K + 1) if k == "3+" else k
    nb = -(K + 1) if k == "3+" else -k
    ordered_cols += [("acc", pb), ("acc", nb), ("n", pb), ("n", nb)]

flip_wide = flip_wide.reindex(columns=pd.MultiIndex.from_tuples(ordered_cols))

# relabel buckets
rename_cols = []
for metric, b in flip_wide.columns:
    if b == (K + 1): lab = "3+"
    elif b == -(K + 1): lab = "-3+"
    else: lab = str(b)
    rename_cols.append((metric, lab))
flip_wide.columns = pd.MultiIndex.from_tuples(rename_cols)

rf4_sorted = flip_wide.sort_values(by=("acc", "-1"), ascending=False).round(2)
rf4_sorted

acc         n       acc         n  \
                                           1    -1   1  -1     2    -2   2   
model         train_years feature_set                                        
xgboost       4           daily         0.64  0.67  36  36  0.50  0.62  30   
random_forest 6           daily         0.81  0.64  36  36  0.53  0.67  30   
xgboost       6           daily         0.64  0.64  36  36  0.57  0.71  30   
                          daily+minute  0.67  0.61  36  36  0.50  0.67  30   
random_forest 4           daily         0.78  0.61  36  36  0.53  0.58  30   
xgboost       4           d+m_w/_lag    0.75  0.56  36  36  0.60  0.67  30   
random_forest 4           d+m_w/_lag    0.78  0.53  36  36  0.60  0.54  30   
xgboost       4           daily+minute  0.69  0.50  36  36  0.60  0.50  30   
              6           d+m_w/_lag    0.64  0.50  36  36  0.43  0.42  30   
random_forest 4           daily+minute  0.81  0.50  36  36  0.60  0.50  30   
              6           d+m_w/_lag    0.86  0.47  36  36  0.63  0.46  30   
xgboost       6           min_w/_lag    0.69  0.44  36  36  0.53  0.38  30   
                          minute        0.67  0.44  36  36  0.60  0.42  30   
              4           min_w/_lag    0.69  0.42  36  36  0.67  0.42  30   
random_forest 6           daily+minute  0.86  0.39  36  36  0.60  0.50  30   
xgboost       4           minute        0.58  0.39  36  36  0.57  0.33  30   
random_forest 6           min_w/_lag    0.78  0.36  36  36  0.57  0.33  30   
              4           minute        0.61  0.36  36  36  0.63  0.42  30   
                          min_w/_lag    0.67  0.33  36  36  0.60  0.38  30   
              6           minute        0.72  0.31  36  36  0.67  0.50  30   

                                             acc         n       acc        \
                                        -2     3    -3   3  -3    3+   -3+   
model         train_years feature_set                                        
xgboost       4           daily         24  0.68  0.57  19  14  0.48  0.65   
random_forest 6           daily         24  0.79  0.57  19  14  0.61  0.61   
xgboost       6           daily         24  0.79  0.43  19  14  0.65  0.57   
                          daily+minute  24  0.53  0.43  19  14  0.50  0.48   
random_forest 4           daily         24  0.74  0.71  19  14  0.61  0.52   
xgboost       4           d+m_w/_lag    24  0.58  0.29  19  14  0.59  0.52   
random_forest 4           d+m_w/_lag    24  0.79  0.43  19  14  0.50  0.52   
xgboost       4           daily+minute  24  0.58  0.43  19  14  0.48  0.52   
              6           d+m_w/_lag    24  0.63  0.50  19  14  0.54  0.70   
random_forest 4           daily+minute  24  0.74  0.29  19  14  0.54  0.48   
              6           d+m_w/_lag    24  0.79  0.29  19  14  0.61  0.57   
xgboost       6           min_w/_lag    24  0.58  0.21  19  14  0.48  0.48   
                          minute        24  0.37  0.36  19  14  0.61  0.48   
              4           min_w/_lag    24  0.68  0.21  19  14  0.57  0.48   
random_forest 6           daily+minute  24  0.74  0.36  19  14  0.63  0.43   
xgboost       4           minute        24  0.42  0.29  19  14  0.48  0.35   
random_forest 6           min_w/_lag    24  0.79  0.14  19  14  0.65  0.43   
              4           minute        24  0.68  0.29  19  14  0.63  0.39   
                          min_w/_lag    24  0.84  0.14  19  14  0.54  0.43   
              6           minute        24  0.68  0.36  19  14  0.59  0.39   

                                         n      
                                        3+ -3+  
model         train_years feature_set           
xgboost       4           daily         46  23  
random_forest 6           daily         46  23  
xgboost       6           daily         46  23  
                          daily+minute  46  23  
random_forest 4           daily         46  23  
xgboost       4           d+m_w/_lag    46  23  
random_for